# Week 17: Vector tiles with tippecanoe and MBTiles

**Track:** Mission GIS Engineer (Advanced)  
**Full primer + quiz:** [https://launchdetect.com/academy/week/17/](https://launchdetect.com/academy/week/17/)  
**Track index:** [https://launchdetect.com/academy/mission-gis-engineer/](https://launchdetect.com/academy/mission-gis-engineer/)

---

_Vector tiles changed web mapping. tippecanoe is the industry-standard generator. This week you take a GeoJSON of 10,000 satellites and produce a smooth, multi-zoom vector tile set._


## Why this week matters

Vector tiles are why modern maps load instantly. A 10,000-feature GeoJSON is unusable directly; the same data as vector tiles is butter-smooth at every zoom. This week is the tile pipeline from generation to serving.


## Learning objectives

By the end of this lab you will be able to:

- Generate vector tiles from GeoJSON with tippecanoe
- Understand MBTiles file format and PMTiles cloud-optimized variant
- Serve tiles via TiTiler or martin
- Configure zoom-dependent tile generalization


## Setup — and why these dependencies

This lab uses: `leafmap[common] requests websockets`. Run the cell below once; Colab installs into the runtime.


In [ ]:
# Install required packages
!pip install -q leafmap[common] requests websockets


## The approach (and why)

Generate a 100-point sample dataset as GeoJSON, install tippecanoe, and produce PMTiles. PMTiles is the modern format because it's served directly from S3 with no tile-server infrastructure.


In [ ]:
# Week 17: vector-tile pipeline preview (full Capstone with tippecanoe in your local env).
import leafmap.foliumap as leafmap

# Demo data: 100 random sub-satellite points
import random
random.seed(42)
pts = [{"type": "Feature",
        "geometry": {"type": "Point", "coordinates": [random.uniform(-180, 180), random.uniform(-60, 60)]},
        "properties": {"sat_id": i, "alt_km": random.randint(400, 800)}}
       for i in range(100)]

fc = {"type": "FeatureCollection", "features": pts}
with open("satellites.geojson", "w") as f:
    import json; json.dump(fc, f)
print(f"Wrote satellites.geojson with {len(pts)} features.")
print("\nTo build vector tiles locally:")
print("  brew install tippecanoe  # or: apt-get install tippecanoe")
print("  tippecanoe -o satellites.pmtiles --maximum-zoom=10 --layer=satellites satellites.geojson")

m = leafmap.Map(center=[0, 0], zoom=2)
m.add_geojson(fc)
m

# TODO: download the full active CelesTrak catalog (~10,000 sats), compute
# current sub-satellite points with skyfield, export GeoJSON, generate vector
# tiles with tippecanoe.


## What just happened — and why it works

tippecanoe takes a GeoJSON, builds a pyramid of small tiles indexed by (z, x, y), and emits a single PMTiles archive. The PMTiles consumer (MapLibre with the PMTiles plugin) range-requests just the tiles in the current viewport.


## Common gotchas

- Tile size sweet spot is <50 KB. Bigger and your frame rate suffers.
- --drop-densest-as-needed is essential for large datasets; without it tippecanoe will refuse to generate tiles where features exceed the per-tile limit.
- tippecanoe is a CLI tool — not pip-installable. Use brew, apt, or build from source.


## Self-check

Verify your solution against these checks before considering the lab complete:

- [ ] Output type matches the expected format (GeoJSON / PNG / table / etc.).
- [ ] No exceptions raised on a clean run.
- [ ] Result is visually plausible — open the map cell, scan the values, sanity-check magnitudes.
- [ ] Quiz on the [week page](https://launchdetect.com/academy/week/17/) — try answering before checking the key.

---

Found a bug or want to contribute an improvement? Open an issue or PR at [github.com/launchdetect/academy-labs](https://github.com/launchdetect/academy-labs).
